# Cats vs Dogs — Binary Image Classification with CNNs

**Daily Challenge Submission**

This notebook covers all 12 steps: data loading, inspection, model design, training with/without augmentation, evaluation, inference, class imbalance handling, artifact saving, and an extension.

---

## Step 1 — Data Loading and Generators

*(Prefilled — copy and execute)*

This block discovers images under `data/cats_dogs/`, infers labels from folder names or filename patterns, performs an 80/20 stratified train/val split, and builds three Keras generators:
- `train_flow` — with augmentation
- `val_flow` — rescale only
- `test_flow` — unlabeled, for inference

In [ ]:
import os, math, re, random
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

np.random.seed(42); tf.random.set_seed(42)

# Paths — adjust DATA_ROOT if needed
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test"  / "test")  if (DATA_ROOT / "test"  / "test").exists()  else (DATA_ROOT / "test")

IMG_HEIGHT, IMG_WIDTH = 180, 180   # reduce to 48 if RAM is limited
BATCH_SIZE = 32
SEED = 1337

def build_df_from_folder(folder: Path, labeled: bool = True):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"No images found under {folder}")
    rows = []
    for f in files:
        if labeled:
            name   = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat", "cats"}: label = "cat"
            elif parent in {"dog", "dogs"}: label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = "dog"
                else: continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2,
    stratify=df_train_full["label"], random_state=SEED
)

# Generators
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

common = dict(target_size=(IMG_HEIGHT, IMG_WIDTH),
              class_mode="binary", batch_size=BATCH_SIZE,
              validate_filenames=False)

train_flow = train_gen.flow_from_dataframe(
    df_tr,  x_col="filepath", y_col="label",
    shuffle=True, seed=SEED, **common)

val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    shuffle=False, **common)

test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=BATCH_SIZE,
    shuffle=False, validate_filenames=False)

print({"train": train_flow.samples, "val": val_flow.samples,
       "test": test_flow.samples, "class_indices": train_flow.class_indices})

---
## Step 2 — Inspect the Data

**TODO — write your analysis here:**

After running the cell below, fill in a short paragraph covering:
- Total images per class (cat / dog) and whether they are balanced
- Potential sources of visual variability (pose, scale, lighting, background)
- What visual cues in the sample grid help distinguish cats from dogs

In [ ]:
import matplotlib.pyplot as plt

# --- Class counts ---
labels_arr = train_flow.labels
unique, counts = np.unique(labels_arr, return_counts=True)
idx_to_class = {v: k for k, v in train_flow.class_indices.items()}
for u, c in zip(unique, counts):
    print(f"{idx_to_class[u]}: {c} images")

# --- Sample grid ---
batch_x, batch_y = next(iter(train_flow))
fig, axes = plt.subplots(3, 4, figsize=(11, 8))
for ax, img, lbl in zip(axes.flat, batch_x, batch_y):
    ax.imshow(img)
    ax.set_title(idx_to_class[int(round(lbl))], fontsize=11)
    ax.axis('off')
fig.suptitle("Sample training images (after augmentation)", fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 3 — Define the Model Architecture

**Architecture description (TODO — fill in your own prose):**

> The model uses 3 convolutional blocks, each with a `Conv2D` layer (32 → 64 → 128 filters, 3×3 kernels, ReLU) followed by `MaxPooling2D` to halve spatial dimensions and reduce parameters. After the final pooling step, `Dropout(0.4)` randomly deactivates 40% of neurons to prevent co-adaptation and reduce overfitting. The flattened feature map feeds a `Dense(256)` hidden layer, then a final `Dense(1, sigmoid)` unit which outputs a probability in [0, 1] — interpreted as P(dog). Binary cross-entropy is the appropriate loss because we have a single Bernoulli target.



In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras import layers

def build_model(img_h=IMG_HEIGHT, img_w=IMG_WIDTH):
    model = Sequential([
        layers.Input(shape=(img_h, img_w, 3)),

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(pool_size=(2, 2)),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(pool_size=(2, 2)),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(pool_size=(2, 2)),

        # Regularisation + classifier head
        layers.Dropout(0.4),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid'),   # binary output
    ], name="cats_dogs_cnn")
    return model

model = build_model()
model.summary()

---
## Step 4 — Optimization Setup

**Justification (TODO — fill in your own prose):**

> **Optimizer:** Adam is chosen for its adaptive per-parameter learning rates and fast empirical convergence on vision tasks.
>
> **Learning rate:** 1e-3 is the Adam default and a safe starting point; too high causes divergence, too low slows progress.
>
> **Batch size:** 32 fits comfortably in CPU/GPU RAM for 180×180 images and provides reasonably low-variance gradient estimates.
>
> **EarlyStopping** monitors `val_loss` (not accuracy) because loss is a smoother, more sensitive signal. `patience=5` allows short plateaus before stopping. `restore_best_weights=True` ensures we keep the best checkpoint.
>
> **ReduceLROnPlateau** halves the LR after 3 stagnant epochs, helping escape flat regions without manual tuning.



In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print("Model compiled. Ready to train.")

---
## Step 5 — Train the Model

In [ ]:
EPOCHS = 30

history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# --- Learning curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history.history['loss'],     label='train loss')
ax1.plot(history.history['val_loss'], label='val loss')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['accuracy'],     label='train acc')
ax2.plot(history.history['val_accuracy'], label='val acc')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.suptitle('Augmented model — training curves', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 6 — Evaluate on Validation Data

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Predict
val_probs = model.predict(val_flow, verbose=0).flatten()
val_preds = (val_probs >= 0.5).astype(int)
val_true  = val_flow.labels

# Loss & accuracy
val_loss, val_acc = model.evaluate(val_flow, verbose=0)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_acc:.4f}\n")

# Classification report
print(classification_report(val_true, val_preds,
      target_names=['cat', 'dog']))

# Confusion matrix
cm = confusion_matrix(val_true, val_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['cat', 'dog'],
            yticklabels=['cat', 'dog'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion matrix — augmented model')
plt.tight_layout()
plt.show()

---
## Step 7 — Inference on Unlabeled Test Set

In [ ]:
# Predict on test set
test_probs = model.predict(test_flow, verbose=1).flatten()

THRESHOLD = 0.5   # justify your choice here
test_preds = ['dog' if p >= THRESHOLD else 'cat' for p in test_probs]

df_out = pd.DataFrame({
    'filepath':   test_flow.filenames,
    'prob_dog':   test_probs.round(4),
    'pred_label': test_preds
})

df_out.to_csv('test_predictions.csv', index=False)
print(f"Saved {len(df_out)} predictions to test_predictions.csv")
df_out.head(10)

**Manual verification strategy (TODO):**

 To sanity-check outputs I would sample ~50 predictions — 25 with high confidence (prob_dog > 0.9 or < 0.1) and 25 near the decision boundary (prob_dog ≈ 0.5) — and visually inspect the corresponding images. High-confidence errors indicate systematic data issues; boundary errors are expected and acceptable.

---
## Step 8 — Baseline vs Augmentation Comparison

In [ ]:
# Baseline generator — no augmentation
baseline_gen = ImageDataGenerator(rescale=1./255)
baseline_flow = baseline_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, validate_filenames=False
)

model_base = build_model()
model_base.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_base = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1)
]

hist_base = model_base.fit(
    baseline_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=callbacks_base,
    verbose=1
)

In [ ]:
# --- Comparison plots ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, key, title in zip(axes, ['loss', 'accuracy'], ['Loss', 'Accuracy']):
    ax.plot(history.history[f'val_{key}'],      label='aug val')
    ax.plot(hist_base.history[f'val_{key}'],    label='baseline val', linestyle='--')
    ax.plot(history.history[key],               label='aug train',    alpha=0.4)
    ax.plot(hist_base.history[key],             label='baseline train', linestyle='--', alpha=0.4)
    ax.set_title(f'Validation {title}')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)

plt.suptitle('Augmented vs Baseline — learning curves', fontsize=13)
plt.tight_layout()
plt.show()

# Quantitative comparison
base_loss, base_acc = model_base.evaluate(val_flow, verbose=0)
aug_loss,  aug_acc  = model.evaluate(val_flow, verbose=0)
print(f"Baseline  — val loss: {base_loss:.4f}, val acc: {base_acc:.4f}")
print(f"Augmented — val loss: {aug_loss:.4f},  val acc: {aug_acc:.4f}")

**Analysis (TODO):**

 The baseline model achieved __% validation accuracy vs __% for the augmented model — a generalization gap of __%. The baseline train/val gap was wider, evidence of overfitting: the model memorized training images rather than learning invariant features. Data augmentation effectively acts as a regularizer, exposing the model to plausible variations (rotations, flips, zoom) it would not otherwise see.

---
## Step 9 — Class Imbalance Handling

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_flow.labels)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_flow.labels
)
class_weight_dict = dict(enumerate(weights))
print("Class weights:", class_weight_dict)

# If weights are close to 1.0, the dataset is balanced — no retraining needed.
# If imbalanced, retrain the augmented model with:
#   model.fit(..., class_weight=class_weight_dict)

**Discussion (TODO):**

> Class weights are [approximately equal / significantly different], indicating the dataset is [balanced / imbalanced]. Class weighting scales the loss contribution of minority-class samples upward, preventing the network from achieving low loss by always predicting the majority class. This typically improves recall for the minority class at a small cost to precision for the majority.

---
## Step 10 — Save Artifacts

In [ ]:
import json, datetime

# Save best model (Keras native format)
model.save('cats_dogs_best_model.keras')
print("Model saved to cats_dogs_best_model.keras")

# Save training config
run_config = {
    "saved_at":        datetime.datetime.now().isoformat(),
    "img_size":        [IMG_HEIGHT, IMG_WIDTH],
    "batch_size":      BATCH_SIZE,
    "max_epochs":      EPOCHS,
    "epochs_trained":  len(history.history['loss']),
    "optimizer":       "adam",
    "learning_rate":   1e-3,
    "augmentation":    True,
    "early_stopping":  {"monitor": "val_loss", "patience": 5},
    "best_val_acc":    float(max(history.history['val_accuracy'])),
    "best_val_loss":   float(min(history.history['val_loss'])),
    "architecture":    "3xConvBlock + Dropout + Dense256 + sigmoid"
}

with open('run_config.json', 'w') as f:
    json.dump(run_config, f, indent=2)
print("Config saved to run_config.json")

with open('run_config.json') as f:
    print(f.read())

**Why save both weights and metadata (TODO):**

> Saving weights alone is insufficient for reproducibility. The metadata records the exact configuration that produced those weights — image size, augmentation policy, optimizer settings, and the achieved metrics. Without it, re-loading a model provides no context for whether it is the correct checkpoint or how it should be used at inference time.

---
## Step 11 — Extension: Transfer Learning with MobileNetV2

**Justification (TODO):**

> Transfer learning with MobileNetV2 pre-trained on ImageNet is chosen because the frozen backbone already encodes low-level features (edges, textures, colours) common to all natural images. Only the small classifier head needs to learn task-specific patterns, so convergence is faster and generalisation is better — especially when training data is limited.

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # freeze backbone

inputs  = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model_tl = Model(inputs, outputs, name="mobilenetv2_transfer")
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model_tl.summary()

In [ ]:
hist_tl = model_tl.fit(
    train_flow,
    validation_data=val_flow,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

tl_loss, tl_acc = model_tl.evaluate(val_flow, verbose=0)
print(f"MobileNetV2 transfer — val loss: {tl_loss:.4f}, val acc: {tl_acc:.4f}")

---
## Step 12 — Deliverables Checklist

Confirm all items are complete before submitting:

- [ ] **Data report** — class counts + sample image grid (Step 2)
- [ ] **Model description** — architecture and optimization rationale in prose (Steps 3–4)
- [ ] **Training curves** — with written interpretation of overfitting / convergence (Step 5)
- [ ] **Validation metrics** — confusion matrix + precision/recall table (Step 6)
- [ ] **test_predictions.csv** — with `filepath`, `prob_dog`, `pred_label` (Step 7)
- [ ] **Augmentation analysis** — comparison curves and generalization gap discussion (Step 8)
- [ ] **Class imbalance discussion** — weights computed and effect explained (Step 9)
- [ ] **Saved model** — `cats_dogs_best_model.keras` (Step 10)
- [ ] **run_config.json** — training metadata (Step 10)
- [ ] **Extension** — transfer learning implemented and justified (Step 11)

---
*End of notebook*